In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

spark.sql("USE CATALOG databricks_7405612194732360")
spark.sql("CREATE SCHEMA IF NOT EXISTS raw_schema")
spark.sql("CREATE VOLUME IF NOT EXISTS raw_schema.raw")

In [0]:
connection_str1 = "******"
eh_sasl1 = f'*****" ;'

connection_str2 = "******"
eh_sasl2 = f'*****" ;'

In [0]:
catalog = "databricks_7405612194732360"
schema  = "raw_schema"
volume  = "raw"
topic1  = "aqiproject"
topic2  = "perdaydata"

city_hour_path    = f"/Volumes/{catalog}/{schema}/{volume}/raw_city_hour"
station_hour_path = f"/Volumes/{catalog}/{schema}/{volume}/raw_station_hour"
hour_checkpoint   = f"/Volumes/{catalog}/{schema}/{volume}/checkpoints/hour"

city_day_path     = f"/Volumes/{catalog}/{schema}/{volume}/raw_city_day"
station_day_path  = f"/Volumes/{catalog}/{schema}/{volume}/raw_station_day"
day_checkpoint    = f"/Volumes/{catalog}/{schema}/{volume}/checkpoints/day"

In [0]:
def kafka_read_stream(topic, sasl):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "capstroneventhub.servicebus.windows.net:9093")
        .option("subscribe", topic)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", sasl)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .load()
        .select(
            col("value").cast("string").alias("raw_payload"),
            col("timestamp").alias("ingestion_datetime"),
            col("topic").alias("source"),
            get_json_object(col("value").cast("string"), "$.origin_type").alias("origin_type")
        )
    )

In [0]:
def write_hourly_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        print(f"[Hourly] Batch {batch_id} — no data, skipping.")
        return
    city_df = batch_df.filter(
        (col("source") == "aqiproject") & (col("origin_type") == "CITY_HOUR")
    )
    if not city_df.isEmpty():
        city_df.write.format("delta").mode("append").save(city_hour_path)
    station_df = batch_df.filter(
        (col("source") == "aqiproject") & (col("origin_type") == "STATION_HOUR")
    )
    if not station_df.isEmpty():
        station_df.write.format("delta").mode("append").save(station_hour_path)
    print(f"[Hourly] Batch {batch_id} written.")


def write_daily_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        print(f"[Daily] Batch {batch_id} — no data, skipping.")
        return
    city_df = batch_df.filter(
        (col("source") == "perdaydata") & (col("origin_type") == "CITY_DAY")
    )
    if not city_df.isEmpty():
        city_df.write.format("delta").mode("append").save(city_day_path)
    station_df = batch_df.filter(
        (col("source") == "perdaydata") & (col("origin_type") == "STATION_DAY")
    )
    if not station_df.isEmpty():
        station_df.write.format("delta").mode("append").save(station_day_path)
    print(f"[Daily] Batch {batch_id} written.")

In [0]:
for q in spark.streams.active:
    q.stop()

hourly_query = (
    kafka_read_stream(topic1, eh_sasl1)
    .writeStream.foreachBatch(write_hourly_batch)
    .option("checkpointLocation", hour_checkpoint)
    .trigger(availableNow=True).start()
)

daily_query = (
    kafka_read_stream(topic2, eh_sasl2)
    .writeStream.foreachBatch(write_daily_batch)
    .option("checkpointLocation", day_checkpoint)
    .trigger(availableNow=True).start()
)

for elapsed in range(10, 60, 10):
    time.sleep(10)
    h = int((hourly_query.lastProgress or {}).get("numInputRows") or 0)
    d = int((daily_query.lastProgress or {}).get("numInputRows") or 0)
    if h > 0 or d > 0:
        break
    if elapsed >= 180:
        for q in spark.streams.active: q.stop()
        raise Exception("No data from Event Hub in 1 minute. Producer may be down.")

hourly_query.awaitTermination()
daily_query.awaitTermination()
print("✅ Ingestion complete.")